# Clean Data Plots and Fits

This notebook explains and plots the two exported files:

- `clean_data.csv`: normal mapped packets. Each row is one accepted 34-byte uplink packet reliably assigned to one normal manual/JSON position window.
- `longer_sampling_per_position.csv`: longer-sampling packets. It contains tail samples plus the requested group 7 rows for `R1352`, `R1189`, and `R1187`.

The main measured columns are `rssi` for signal strength and `snr` for signal-to-noise ratio. The main mapping columns are `structure_id`, `manual_group`, `x_m`, and the interval timestamps used to assign a packet to a position.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = Path("outputs/tests_gh_25_06")
clean_data = pd.read_csv(DATA_DIR / "clean_data.csv")
longer_sampling = pd.read_csv(DATA_DIR / "longer_sampling_per_position.csv")

time_cols = ["time_local", "registered_start_time", "interval_start_time", "interval_end_time"]
for frame in (clean_data, longer_sampling):
    for col in time_cols:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")

clean_data["mapped_row_id"] = np.arange(len(clean_data))
longer_sampling["mapped_row_id"] = np.arange(len(longer_sampling))

print(f"clean_data: {clean_data.shape[0]} rows, {clean_data.shape[1]} columns")
print(f"longer_sampling_per_position: {longer_sampling.shape[0]} rows, {longer_sampling.shape[1]} columns")

## Structure of the Files

Each row is one measured packet. The important column groups are:

- Actual mapped row: `mapped_row_id`, `source_file`, `time_local`, `fcnt`, `type`, `size`, `structure_id`, `manual_group`, `manual_group_status`, `manual_group_reliable`.
- Distance / position: `x_m` is the physical position used for plotting and fitting. It was constructed from the mapped group as `(manual_group - 5) * 15`, with group 12 shifted by `+3.5 m`. This gives a signed position. The fit below uses `abs(x_m)` because path loss depends on distance magnitude.
- Mapping interval: `registered_start_time`, `interval_start_time`, `interval_end_time`, `is_final_group`, `is_extra_tail_group` describe the time window that accepted the packet.
- Signal measurements: `rssi` is received signal strength in dBm. `snr` is signal-to-noise ratio in dB.

`longer_sampling_per_position.csv` also has `longer_sampling_source`, with `tail` for tail samples and `group_7` for the added group 7 samples.

In [ ]:
key_cols = [
    "mapped_row_id", "source_file", "time_local", "fcnt", "type", "size",
    "structure_id", "manual_group", "manual_group_status", "manual_group_reliable",
    "x_m", "registered_start_time", "interval_start_time", "interval_end_time",
    "is_final_group", "is_extra_tail_group", "rssi", "snr", "longer_sampling_source",
]

display(clean_data[[c for c in key_cols if c in clean_data.columns]].head(12))

clean_position_summary = (
    clean_data.groupby(["structure_id", "manual_group", "x_m"], dropna=False)
    .agg(rows=("mapped_row_id", "count"), mean_rssi=("rssi", "mean"), std_rssi=("rssi", "std"), mean_snr=("snr", "mean"), std_snr=("snr", "std"))
    .reset_index()
    .sort_values(["structure_id", "manual_group"])
)
display(clean_position_summary.head(20))

display(longer_sampling["longer_sampling_source"].value_counts().rename_axis("source").to_frame("rows"))
display(longer_sampling[[c for c in key_cols if c in longer_sampling.columns]].head(12))

## Physically Motivated Fit

For RSSI, a standard first model is log-distance path loss:

`RSSI(d) = A - 10 n log10(d / d0)`

Here `d = abs(x_m)` with a floor at `1 m` to avoid `log10(0)`. `A` is the fitted level near `1 m`, and `n` is the path-loss exponent. Free space is often near `n = 2`; larger values suggest stronger attenuation or obstructions.

For SNR, the same log-distance curve is used as an empirical trend model. It is less fundamental than RSSI because SNR also depends on noise/interference.

Fit quality is summarized by `R2`, `RMSE`, number of rows `N`, and slope. Higher `R2` and lower `RMSE` mean distance explains the data better.

In [ ]:
def fit_log_distance(df: pd.DataFrame, metric: str, min_distance_m: float = 1.0) -> dict:
    fit_df = df[["x_m", metric]].dropna().copy()
    fit_df["distance_m"] = fit_df["x_m"].abs().clip(lower=min_distance_m)
    x = np.log10(fit_df["distance_m"].to_numpy())
    y = fit_df[metric].to_numpy()
    slope, intercept, r_value, p_value, stderr = stats.linregress(x, y)
    y_hat = intercept + slope * x
    rmse = float(np.sqrt(np.mean((y - y_hat) ** 2)))
    return {
        "metric": metric,
        "n_rows": len(fit_df),
        "intercept": float(intercept),
        "slope": float(slope),
        "r2": float(r_value ** 2),
        "p_value": float(p_value),
        "stderr": float(stderr),
        "rmse": rmse,
        "path_loss_n": float(-slope / 10) if metric == "rssi" else np.nan,
    }


def add_fit_panel(ax, df: pd.DataFrame, metric: str, color: str):
    fit = fit_log_distance(df, metric)
    plot_df = df[["x_m", metric, "structure_id", "manual_group"]].dropna().copy()
    ax.scatter(plot_df["x_m"], plot_df[metric], s=18, alpha=0.28, color=color, label="packet rows")

    summary = plot_df.groupby("x_m", dropna=False).agg(mean=(metric, "mean"), std=(metric, "std"), n=(metric, "count")).reset_index().sort_values("x_m")
    ax.errorbar(summary["x_m"], summary["mean"], yerr=summary["std"], fmt="o", color="black", ecolor="0.65", capsize=3, label="position mean +/- std")

    x_grid = np.linspace(plot_df["x_m"].min(), plot_df["x_m"].max(), 300)
    d_grid = np.clip(np.abs(x_grid), 1.0, None)
    y_grid = fit["intercept"] + fit["slope"] * np.log10(d_grid)
    ax.plot(x_grid, y_grid, color="crimson", linewidth=2.0, label="log-distance fit")

    details = f"N={fit['n_rows']}\nR2={fit['r2']:.3f}\nRMSE={fit['rmse']:.2f} dB\nslope={fit['slope']:.2f} dB/dec"
    if metric == "rssi":
        details += f"\npath-loss n={fit['path_loss_n']:.2f}"
    ax.text(0.02, 0.03, details, transform=ax.transAxes, va="bottom", ha="left", fontsize=9, bbox=dict(facecolor="white", alpha=0.82, edgecolor="0.8"))
    ax.set_xlabel("Position x_m (m); fit uses abs(x_m)")
    ax.set_ylabel(metric.upper())
    ax.set_title(f"{metric.upper()} per mapped row with log-distance fit")
    ax.grid(True, axis="y", alpha=0.25)
    ax.grid(False, axis="x")
    return fit


fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
rssi_fit = add_fit_panel(axes[0], clean_data, "rssi", "tab:blue")
snr_fit = add_fit_panel(axes[1], clean_data, "snr", "tab:green")
axes[0].legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

fit_summary = pd.DataFrame([rssi_fit, snr_fit])
display(fit_summary)

## Fit Residuals

Residuals show where the physical model misses. If residuals are randomly scattered around zero, the fit is adequate. If residuals form structure by `x_m`, `structure_id`, or group, then distance alone is not enough and other physical effects are important.

In [ ]:
residual_df = clean_data[["structure_id", "manual_group", "x_m", "rssi", "snr"]].dropna().copy()
for fit in (rssi_fit, snr_fit):
    metric = fit["metric"]
    d = residual_df["x_m"].abs().clip(lower=1.0)
    residual_df[f"{metric}_fit"] = fit["intercept"] + fit["slope"] * np.log10(d)
    residual_df[f"{metric}_residual"] = residual_df[metric] - residual_df[f"{metric}_fit"]

residual_summary = residual_df.groupby(["structure_id", "manual_group", "x_m"], dropna=False).agg(rows=("rssi", "count"), mean_rssi_residual=("rssi_residual", "mean"), mean_snr_residual=("snr_residual", "mean")).reset_index().sort_values(["x_m", "structure_id"])
display(residual_summary.head(30))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].scatter(residual_df["x_m"], residual_df["rssi_residual"], s=16, alpha=0.35, color="tab:blue")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("RSSI fit residuals")
axes[0].set_xlabel("x_m (m)")
axes[0].set_ylabel("RSSI residual (dB)")
axes[1].scatter(residual_df["x_m"], residual_df["snr_residual"], s=16, alpha=0.35, color="tab:green")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("SNR fit residuals")
axes[1].set_xlabel("x_m (m)")
axes[1].set_ylabel("SNR residual (dB)")
plt.tight_layout()
plt.show()

## Longer-Sampling Distribution Study

For repeated measurements at a fixed position, RSSI and SNR often look approximately normal around a local mean because many small effects add together. Wireless measurements can also have heavier tails due to multipath, interference, timing, or orientation changes.

For each longer-sampling position, the code fits a normal distribution and a Student-t distribution. Lower AIC is better. If `delta_aic_t_minus_normal < 0`, the Student-t model is preferred; otherwise the normal model is preferred. Shapiro p-values are included as a rough normality check.

In [ ]:
def distribution_fit_table(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    rows = []
    group_cols = ["longer_sampling_source", "structure_id", "manual_group", "x_m"]
    for key, group in df.groupby(group_cols, dropna=False):
        values = group[metric].dropna().to_numpy()
        if len(values) < 3:
            continue
        source, structure_id, manual_group, x_m = key
        mu, sigma = stats.norm.fit(values)
        sigma = max(float(sigma), 1e-9)
        normal_loglik = float(np.sum(stats.norm.logpdf(values, mu, sigma)))
        normal_aic = 2 * 2 - 2 * normal_loglik
        try:
            t_df, t_loc, t_scale = stats.t.fit(values)
            t_scale = max(float(t_scale), 1e-9)
            t_loglik = float(np.sum(stats.t.logpdf(values, t_df, loc=t_loc, scale=t_scale)))
            t_aic = 2 * 3 - 2 * t_loglik
        except Exception:
            t_df, t_loc, t_scale, t_aic = np.nan, np.nan, np.nan, np.nan
        shapiro_p = stats.shapiro(values).pvalue if 3 <= len(values) <= 5000 else np.nan
        rows.append({
            "metric": metric,
            "longer_sampling_source": source,
            "structure_id": structure_id,
            "manual_group": manual_group,
            "x_m": x_m,
            "n": len(values),
            "mean": float(np.mean(values)),
            "std": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
            "median": float(np.median(values)),
            "normal_mu": float(mu),
            "normal_sigma": float(sigma),
            "normal_aic": float(normal_aic),
            "t_df": float(t_df),
            "t_loc": float(t_loc),
            "t_scale": float(t_scale),
            "t_aic": float(t_aic),
            "delta_aic_t_minus_normal": float(t_aic - normal_aic) if np.isfinite(t_aic) else np.nan,
            "shapiro_p": float(shapiro_p),
        })
    return pd.DataFrame(rows).sort_values(["metric", "longer_sampling_source", "structure_id", "manual_group"])


distribution_summary = pd.concat([distribution_fit_table(longer_sampling, "rssi"), distribution_fit_table(longer_sampling, "snr")], ignore_index=True)
display(distribution_summary)

model_preference = (
    distribution_summary.assign(preferred=np.where(distribution_summary["delta_aic_t_minus_normal"] < 0, "student_t", "normal"))
    .groupby(["metric", "preferred"])
    .size()
    .rename("positions")
    .reset_index()
)
display(model_preference)

In [ ]:
def plot_longer_sampling_distributions(df: pd.DataFrame, metric: str):
    group_cols = ["longer_sampling_source", "structure_id", "manual_group", "x_m"]
    groups = [(key, group[metric].dropna().to_numpy()) for key, group in df.groupby(group_cols, dropna=False) if group[metric].notna().sum() >= 3]
    ncols = 3
    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.4 * nrows), squeeze=False)
    axes = axes.ravel()
    for ax, (key, values) in zip(axes, groups):
        source, structure_id, manual_group, x_m = key
        ax.hist(values, bins="auto", density=True, alpha=0.62, color="tab:blue" if metric == "rssi" else "tab:green", edgecolor="white")
        mu, sigma = stats.norm.fit(values)
        sigma = max(float(sigma), 1e-9)
        x_grid = np.linspace(values.min() - 2 * sigma, values.max() + 2 * sigma, 250)
        ax.plot(x_grid, stats.norm.pdf(x_grid, mu, sigma), color="black", linewidth=1.8, label="normal fit")
        ax.axvline(np.mean(values), color="crimson", linewidth=1.2, linestyle="--", label="mean")
        ax.set_title(f"{source} {structure_id} G{int(manual_group)} x={x_m:g} m\nN={len(values)}, mean={np.mean(values):.1f}, std={np.std(values, ddof=1):.1f}")
        ax.set_xlabel(metric.upper())
        ax.set_ylabel("Density")
        ax.grid(True, axis="y", alpha=0.25)
        ax.grid(False, axis="x")
    for ax in axes[len(groups):]:
        ax.axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")
    fig.suptitle(f"Longer-sampling {metric.upper()} distributions by mapped row/position", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


plot_longer_sampling_distributions(longer_sampling, "rssi")
plot_longer_sampling_distributions(longer_sampling, "snr")

## Distribution Interpretation

Use the `distribution_summary` table together with the histograms:

- A roughly symmetric histogram with lower normal AIC means a normal model is reasonable.
- A lower Student-t AIC means heavier tails are likely, so occasional strong or weak packets occur more often than Gaussian noise would predict.
- If `N` is small, treat model choice cautiously. The longer-sampling rows help because larger `N` makes this distribution study more meaningful.

A small standard deviation and normal-looking histogram indicate stable reception at that position. Large spread or heavy tails suggest transient propagation, multipath, interference, or packet-level variability.